# Principal Component Analysis (PCA)

PCA is an unsupervised dimensionality reduction method. It projects data onto the directions of **maximum variance** — the principal components — enabling high-dimensional data to be visualised and compressed.

**Dataset:** Iris (4 features → 2 principal components)

**Mathematical Core:**  
Centre the data, compute the covariance matrix, and take the top eigenvectors (via SVD):  
$$\mathbf{X}_{\text{centered}} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^T \quad \Rightarrow \quad \text{PC}_k = \text{column } k \text{ of } \mathbf{V}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
import sys
sys.path.insert(0, r"/Jana CMOR/2026_Data_Science_and_Machine_Learning/src/rice_ml/unsupervised_learning")
from rice_ml.unsupervised_learning.pca import pca
np.random.seed(42)
print("Imports complete")

## Load & Explore the Dataset

We use the Iris dataset — 4 numeric features across 3 species. All features are in centimetres but on slightly different scales, so standardisation is needed before PCA.

In [ ]:
data = load_iris()
X, y = data.data, data.target
feature_names = data.feature_names
class_names   = data.target_names

print(f"Shape: {X.shape} | Classes: {list(class_names)}")
print(f"Feature ranges: {X.min(axis=0).round(2)} to {X.max(axis=0).round(2)}")

## Exploratory Data Analysis

Pairwise scatter plots confirm that petal features are highly discriminative, while sepal features overlap between classes.

In [ ]:
colors = ['#e63946', '#457b9d', '#2a9d8f']
fig, axes = plt.subplots(4, 4, figsize=(13, 13))
for row in range(4):
    for col in range(4):
        ax = axes[row, col]
        if row == col:
            for cls in range(3):
                ax.hist(X[y == cls, col], bins=12, alpha=0.6, color=colors[cls], edgecolor='k', lw=0.3)
            ax.set_title(feature_names[col].replace(" (cm)", ""), fontsize=8)
        else:
            for cls in range(3):
                mask = y == cls
                ax.scatter(X[mask, col], X[mask, row], color=colors[cls], alpha=0.55,
                           edgecolors='k', linewidths=0.3, s=20)
        if col == 0:
            ax.set_ylabel(feature_names[row].replace(" (cm)", ""), fontsize=7)
        if row == 3:
            ax.set_xlabel(feature_names[col].replace(" (cm)", ""), fontsize=7)
        ax.tick_params(labelsize=7)

handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c,
                       markersize=8, label=n) for c, n in zip(colors, class_names)]
fig.legend(handles=handles, loc='upper right', fontsize=10)
plt.suptitle("Pairwise Feature Scatter / Diagonal Histograms", fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Standardise the Data

PCA is scale-sensitive — a feature measured in mm would dominate one in cm.  
We standardise so each feature has mean=0 and variance=1.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Mean per feature (should be ~0): {X_scaled.mean(axis=0).round(4)}")
print(f"Std per feature (should be ~1):  {X_scaled.std(axis=0).round(4)}")

## Fit PCA — All Components & Scree Plot

The scree plot shows how much variance each principal component explains. We look for the "elbow" where the curve flattens.

In [ ]:
model_full = pca(n_components=None)
model_full.fit(X_scaled)

evr = model_full.explained_variance_ratio_
print("Explained variance ratio per component:")
for i, r in enumerate(evr):
    print(f"  PC{i+1}: {r:.4f}  ({r*100:.2f}%)  cumulative: {np.sum(evr[:i+1])*100:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scree plot
axes[0].bar(range(1, 5), evr * 100, color=['#e63946', '#457b9d', '#2a9d8f', '#e9c46a'],
            edgecolor='k', alpha=0.85)
axes[0].plot(range(1, 5), np.cumsum(evr) * 100, 'ko-', linewidth=2, markersize=8, label='Cumulative %')
axes[0].axhline(95, color='tomato', linestyle='--', linewidth=1.5, label='95% threshold')
axes[0].set_xlabel("Principal Component", fontsize=12)
axes[0].set_ylabel("Explained Variance (%)", fontsize=12)
axes[0].set_title("Scree Plot", fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, axis='y', linestyle='--', alpha=0.5)

# Eigenvalue bar chart
axes[1].bar(range(1, 5), model_full.explained_variance_, color='steelblue', edgecolor='k', alpha=0.8)
axes[1].set_xlabel("Principal Component", fontsize=12)
axes[1].set_ylabel("Eigenvalue (Explained Variance)", fontsize=12)
axes[1].set_title("Eigenvalues by Component", fontsize=13, fontweight='bold')
axes[1].grid(True, axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Project to 2 Principal Components

Two components capture ~95.8% of total variance while reducing dimensionality by 50%.  
The 2D projection clearly separates the three Iris species.

In [ ]:
model_2d = pca(n_components=2)
X_2d = model_2d.fit_transform(X_scaled)
print(f"Original: {X_scaled.shape}  →  Reduced: {X_2d.shape}")
print(f"Variance explained: {model_2d.explained_variance_ratio_.sum()*100:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 2D scatter
for cls, color, name in zip(range(3), ['#e63946', '#457b9d', '#2a9d8f'], class_names):
    mask = y == cls
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1], color=color, label=name,
                    alpha=0.75, edgecolors='k', linewidths=0.4, s=60)
axes[0].set_xlabel(f"PC1 ({model_2d.explained_variance_ratio_[0]*100:.1f}% variance)", fontsize=12)
axes[0].set_ylabel(f"PC2 ({model_2d.explained_variance_ratio_[1]*100:.1f}% variance)", fontsize=12)
axes[0].set_title("Iris Projected onto 2 Principal Components", fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

# Biplot — show feature loadings as arrows
for i, (name, loading) in enumerate(zip(feature_names, model_2d.components_.T)):
    axes[1].annotate('', xy=loading * 3.5, xytext=(0, 0),
                     arrowprops=dict(arrowstyle='->', color='tomato', lw=2.5))
    axes[1].text(loading[0] * 3.8, loading[1] * 3.8,
                 name.replace(" (cm)", ""), fontsize=9, color='darkred', fontweight='bold')
for cls, color, name in zip(range(3), ['#e63946', '#457b9d', '#2a9d8f'], class_names):
    mask = y == cls
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1], color=color, label=name,
                    alpha=0.4, s=40, edgecolors='k', linewidths=0.3)
axes[1].set_xlabel("PC1", fontsize=12); axes[1].set_ylabel("PC2", fontsize=12)
axes[1].set_title("Biplot — Scores + Feature Loadings", fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Reconstruction Error vs Components

Adding more components reduces reconstruction error. We visualise the trade-off between dimensionality and information loss.

In [ ]:
n_comp_range = range(1, 5)
errors = []
var_cumulative = []

for n in n_comp_range:
    m = pca(n_components=n)
    m.fit(X_scaled)
    err = m.reconstruction_error(X_scaled)
    errors.append(err)
    var_cumulative.append(m.explained_variance_ratio_.sum() * 100)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(list(n_comp_range), errors, 'o-', color='tomato', linewidth=2.5, markersize=9)
for n, e in zip(n_comp_range, errors):
    axes[0].annotate(f"{e:.4f}", (n, e), textcoords='offset points', xytext=(5, 5), fontsize=9)
axes[0].set_xlabel("Number of Components", fontsize=12)
axes[0].set_ylabel("Mean Squared Reconstruction Error", fontsize=12)
axes[0].set_title("Reconstruction Error vs Components", fontsize=13, fontweight='bold')
axes[0].set_xticks(list(n_comp_range))
axes[0].grid(True, linestyle='--', alpha=0.5)

axes[1].bar(list(n_comp_range), var_cumulative, color='#457b9d', edgecolor='k', alpha=0.8)
axes[1].axhline(95, color='tomato', linestyle='--', lw=1.5, label='95% threshold')
for n, v in zip(n_comp_range, var_cumulative):
    axes[1].text(n, v + 0.5, f"{v:.1f}%", ha='center', fontsize=10, fontweight='bold')
axes[1].set_xlabel("Number of Components", fontsize=12)
axes[1].set_ylabel("Cumulative Variance Explained (%)", fontsize=12)
axes[1].set_title("Cumulative Variance Explained", fontsize=13, fontweight='bold')
axes[1].set_xticks(list(n_comp_range))
axes[1].set_ylim(0, 110)
axes[1].legend(fontsize=10)
axes[1].grid(True, axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f"{'Components':>12} | {'Variance':>10} | {'Recon. Error':>14}")
print("-" * 44)
for n, v, e in zip(n_comp_range, var_cumulative, errors):
    print(f"{n:>12} | {v:>9.2f}% | {e:>14.6f}")

## Component Loadings

Loadings reveal each original feature's contribution to each principal component. High absolute loadings indicate features that drive that component.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
short_names = ['Sepal L', 'Sepal W', 'Petal L', 'Petal W']
colors_load = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a']

for i, ax in enumerate(axes):
    loadings = model_2d.components_[i]
    bar_cols = ['#457b9d' if l > 0 else '#e63946' for l in loadings]
    bars = ax.bar(short_names, loadings, color=bar_cols, edgecolor='k', alpha=0.85)
    ax.axhline(0, color='k', linewidth=0.8)
    for bar, val in zip(bars, loadings):
        ax.text(bar.get_x() + bar.get_width()/2,
                val + 0.01 * np.sign(val), f"{val:.3f}",
                ha='center', fontsize=10, fontweight='bold')
    ax.set_ylabel("Loading", fontsize=11)
    ax.set_title(f"PC{i+1} Loadings ({model_2d.explained_variance_ratio_[i]*100:.1f}% var)",
                 fontsize=12, fontweight='bold')
    ax.grid(True, axis='y', linestyle='--', alpha=0.5)
    ax.set_ylim(-0.8, 0.8)

plt.suptitle("Feature Loadings for PC1 and PC2", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Key Takeaways

- Two PCs capture **~95.8%** of Iris variance — near-lossless reduction from 4→2 dimensions.
- *Setosa* is clearly linearly separable in PC space; *Versicolor* and *Virginica* overlap slightly.
- PC1 is dominated by **petal size** (large positive loading) — the most discriminative axis.
- PC2 is driven by **sepal width** — explaining orthogonal variance not captured by PC1.
- Reconstruction error drops sharply from 1→2 components, then flattens — the optimal cut-off.